In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/adaptive-learning/data/raw/StudentsPerformance.csv')
print('Missing values per column:')
print(df.isnull().sum())


Missing values per column:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64


In [3]:
from scipy import stats
import numpy as np

score_cols = ['math score', 'reading score', 'writing score']
z_scores = np.abs(stats.zscore(df[score_cols]))
outlier_mask = (z_scores > 3).any(axis=1)
print(f'Number of outlier rows: {outlier_mask.sum()}')
print(df[outlier_mask])


Number of outlier rows: 7
     gender race/ethnicity parental level of education         lunch  \
17   female        group B            some high school  free/reduced   
59   female        group C            some high school  free/reduced   
76     male        group E            some high school      standard   
327    male        group A                some college  free/reduced   
596    male        group B                 high school  free/reduced   
787  female        group B                some college      standard   
980  female        group B                 high school  free/reduced   

    test preparation course  math score  reading score  writing score  
17                     none          18             32             28  
59                     none           0             17             10  
76                     none          30             26             22  
327                    none          28             23             19  
596                    none          

In [4]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Removed {before - after} duplicate rows. Rows remaining: {after}')


Removed 0 duplicate rows. Rows remaining: 1000


In [5]:
df['total_score'] = df['math score'] + df['reading score'] + df['writing score']

def assign_level(total):
    if total <= 149:   return 'Low'
    elif total <= 199: return 'Developing'
    elif total <= 249: return 'Proficient'
    else:              return 'Advanced'

df['performance_category'] = df['total_score'].apply(assign_level)
print(df['performance_category'].value_counts(normalize=True).round(3))


performance_category
Proficient    0.417
Developing    0.341
Advanced      0.139
Low           0.103
Name: proportion, dtype: float64


In [7]:
fcl_mapping = {
    'Low':        {'fcl_range': '1-4',  'description': 'Foundational comprehension, Grades 1-4'},
    'Developing': {'fcl_range': '5-7',  'description': 'Elementary comprehension, Grades 5-7'},
    'Proficient': {'fcl_range': '8-10', 'description': 'Secondary comprehension, Grades 8-10'},
    'Advanced':   {'fcl_range': '11-T', 'description': 'Advanced/Tertiary comprehension'},
}

import json
with open('/content/drive/MyDrive/adaptive-learning/models/fcl_mapping.json', 'w') as f:
    json.dump(fcl_mapping, f, indent=2)
print('FCL mapping saved.')


FCL mapping saved.


In [8]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['performance_category', 'total_score'])
y = df['performance_category']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')


Train: 700 | Val: 150 | Test: 150


In [9]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

nominal_cols = ['gender', 'race/ethnicity', 'lunch', 'test preparation course']
ordinal_cols = ['parental level of education']
ordinal_order = [['some high school', 'high school', 'some college',
                   "associate's degree", "bachelor's degree", "master's degree"]]

preprocessor = ColumnTransformer(transformers=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
    ('ordinal', OrdinalEncoder(categories=ordinal_order), ordinal_cols),
], remainder='passthrough')

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc   = preprocessor.transform(X_val)
X_test_enc  = preprocessor.transform(X_test)


In [10]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train_enc, y_train, random_state=42)
mi_df = pd.DataFrame({'feature': range(len(mi_scores)), 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)
print(mi_df.head(15))


    feature  mi_score
13       13  0.925558
14       14  0.921260
12       12  0.685821
9         9  0.052526
10       10  0.029230
8         8  0.025836
0         0  0.024416
5         5  0.018987
2         2  0.016291
6         6  0.008093
11       11  0.007068
3         3  0.000000
4         4  0.000000
1         1  0.000000
7         7  0.000000


In [11]:
print('Class distribution in training set:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_enc, y_train)
print('After SMOTE:')
print(pd.Series(y_train_balanced).value_counts(normalize=True).round(3))


Class distribution in training set:
performance_category
Proficient    0.417
Developing    0.341
Advanced      0.139
Low           0.103
Name: proportion, dtype: float64
After SMOTE:
performance_category
Advanced      0.25
Proficient    0.25
Developing    0.25
Low           0.25
Name: proportion, dtype: float64


In [12]:
import joblib

DRIVE_PATH = '/content/drive/MyDrive/adaptive-learning/'

joblib.dump(preprocessor, DRIVE_PATH + 'models/preprocessor.joblib')

pd.DataFrame(X_train_balanced).to_csv(DRIVE_PATH + 'data/X_train.csv', index=False)
pd.Series(y_train_balanced).to_csv(DRIVE_PATH + 'data/y_train.csv', index=False)
pd.DataFrame(X_val_enc).to_csv(DRIVE_PATH + 'data/X_val.csv', index=False)
pd.Series(y_val).to_csv(DRIVE_PATH + 'data/y_val.csv', index=False)
pd.DataFrame(X_test_enc).to_csv(DRIVE_PATH + 'data/X_test.csv', index=False)
pd.Series(y_test).to_csv(DRIVE_PATH + 'data/y_test.csv', index=False)

# Save original (unencoded) X_test for fairness analysis in Week 2
X_test.to_csv(DRIVE_PATH + 'data/X_test_original.csv', index=False)

print('All files saved successfully.')


All files saved successfully.
